In [1]:

import numpy as np 
from langchain_groq import ChatGroq 
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter 
from langchain.vectorstores import Chroma
from dotenv import load_dotenv 
from langchain_core.output_parsers import StrOutputParser
load_dotenv()

True

In [2]:
pdf_path='sample.pdf'
loader=PyPDFLoader(pdf_path)
documents=loader.load()

In [3]:
print('number of pages in the report are ',len(documents))

number of pages in the report are  55


In [4]:
#now lets convert these docs into a text and then we will split it 
# print(documents[0].page_content)
pdf_text=""

In [5]:
for doc in documents:
    pdf_text+=doc.page_content

In [7]:
pdf_text

'i \n \nData Science and Machine Learning \nA \nIndustrial/Field Project Report \nsubmitted \nin partial fulfilment \nfor the award of the degree of \nBachelor of Technology \nin department of Computer Science \nwith specialization in Computer Science \n \nSupervisor        Submitted by \nMr. Pratap Singh Patwal      Aashish Kumar Saini \nHOD, CSE       22ELDCS003 \nLIET - ALWAR \n   \n \n \n \n \n \n \n         Department of Computer Science and Engineering  \n  Laxmi Devi Institute of Engineering and Technology, Alwar  \n    Bikaner Technical University  \n     Dec – 2024ii \n \nCandidate’s Declaration \n \nI hereby declare that the Industrial/Field Project, which is being intern in the field, \nentitled “Data Science and Machine Learning” in partial fulfilment for the award \nof Degree of “ Bachelor of Technology” in Department of Computer Science & \nEngineering with Specialization in Computer Science Engineering , and submitted \nto the Department  of Computer Science & Engineerin

In [ ]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

In [9]:
chunks=text_splitter.split_text(pdf_text)

In [10]:
len(chunks)

90

In [11]:
#now we required a embedding model for stroing all the chunks as embedding in the vector db 
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embedding_model=GoogleGenerativeAIEmbeddings(model='models/text-embedding-004')

In [12]:
embd=embedding_model.embed_query(text='hi my name is aashish')

In [13]:
len(embd) #768 dimensional embedding vector 

768

In [14]:
#lets store the data in the vector db 
vector_db=Chroma.from_texts(texts=chunks,embedding=embedding_model) 

In [45]:
#now lets add a llm model that we will use for generatiing the reponse from the document
llm=ChatGroq(model='Llama-3.3-70b-Versatile')

In [46]:
# llm.invoke('hello') #working 

In [47]:
#lets design a prompt template for rag that we will use to give the context to the llm 
from langchain.prompts import PromptTemplate
prop_t=PromptTemplate.from_template("""" You are a helpful AI assistant. And you have to generate teh reposponse
                using the following content 
                If the content is not related to the question then simply say 'I am sorry I dont have that information'
                
                query -- {query} 
                
                
                context -- {context}""")

In [48]:
llm_chain=prop_t|llm|StrOutputParser()

In [15]:
#now lets test our rag system
chunk_retriever=vector_db.as_retriever(search_type='similarity',kwargs={'k':5})

In [16]:
output=chunk_retriever.invoke('what is the machine learning ')

{}

In [ ]:
for text in output:
    print(text)
#now we know that our code is working fine 


page_content='isualization: Tableau, Power BI, Matplotlib, Seaborn 
• Machine Learning Frameworks: Scikit-learn, TensorFlow, PyTorch 
• Big Data Tools: Hadoop, Spark 
• Cloud Platforms: AWS, Google Cloud, Microsoft Azure7 
 
3. INTRODUCTION TO MACHINE LEARNING 
 
3.1 What is Machine Learning? 
Machine learning is a branch of artificial intelligence that enables algorithms to 
uncover hidden patterns within datasets, allowing them to make predictions on new, 
similar data without explicit programming for each task. Traditional machine 
learning combines data with statistical tools to predict outputs, yielding actionable 
insights. This technology finds applications in diverse fields such as image and 
speech recognition, natural language processing, recommendation systems, fraud 
detection, portfolio optimization, and automating tasks. 
For instance, recommender systems use historical data to personalize suggestions. 
Netflix, for example, employs collaborative and content -based filter

In [62]:
#lets create a function for rag qna 
from IPython.display import Markdown,display
def rag_qa(query):
    relevant_docs=chunk_retriever.invoke(query)
    text=[doc.page_content for doc in relevant_docs]
    
    response=llm_chain.invoke({'query':query,'context':' '.join(text)})
    # print(response)
    return response
    

In [67]:
while True: 
    user_input=input('Enter You Query Here: ')
    display(Markdown(f"Human: {user_input}"))
    if user_input.lower() in ['exit','stop']:
        break 
    else:
        
        response=rag_qa(user_input)
        display(Markdown(f"Assistant: {response}"))

Human: what is ml

Assistant: I am sorry I don't have that information. The provided context is about Linear Regression, but it does not mention what ML (Machine Learning) is.

Human: what is ML

Assistant: Machine learning is a branch of artificial intelligence that enables algorithms to uncover hidden patterns within datasets, allowing them to make predictions on new, similar data without explicit programming for each task. Traditional machine learning combines data with statistical tools to predict outputs, yielding actionable insights. This technology finds applications in diverse fields such as image and speech recognition, natural language processing, recommendation systems, fraud detection, portfolio optimization, and automating tasks.

Human: waht are the types of ml

Assistant: The types of Machine Learning (ML) are:

1. **Supervised Machine Learning**: This type of ML involves training machines using labeled datasets, where the machine is trained on input and corresponding output, and then predicts the output using a test dataset.
2. **Unsupervised Machine Learning**: This type of ML involves training machines on unlabeled datasets, where the machine identifies patterns or relationships in the data without prior knowledge of the output.
3. **Semi-Supervised Machine Learning**: This type of ML combines elements of both supervised and unsupervised learning, where the machine is trained on a small amount of labeled data and a large amount of unlabeled data.
4. **Reinforcement Machine Learning**: This type of ML involves training machines to make decisions based on rewards or penalties, where the machine learns to take actions to maximize rewards or minimize penalties.

These four types of Machine Learning can be applied to various problems, including Regression, Classification, Forecasting, Clustering, and Associations.

Human: exit

In [76]:
#now lets improve the performance of the rag system 
#firstly we will pass the query to the llm and then we will tell our llm to generate a query 
#that will help to get the best chunks for the vector store 

from pydantic import BaseModel ,Field
from typing import Annotated

class Schema1(BaseModel):
    query: Annotated[str,Field(...,description='Please elaborate this given query in such a way that by it we can retrieve the best similar documents from vector store ')]
    

In [77]:
llm1=ChatGroq(model='Llama-3.3-70b-Versatile')
query_extender=llm1.with_structured_output(Schema1)

In [81]:
query_extender.invoke('tell me ml and aslo its types').query

'Machine learning types and details'

In [82]:
#let's create a new function
def rag_qa(query):
    query=query_extender.invoke(query).query
    relevant_docs=chunk_retriever.invoke(query)
    text=[doc.page_content for doc in relevant_docs]
    
    response=llm_chain.invoke({'query':query,'context':' '.join(text)})
    # print(response)
    return response


while True: 
    user_input=input('Enter You Query Here: ')
    display(Markdown(f"Human: {user_input}"))
    if user_input.lower() in ['exit','stop']:
        break 
    else:
        
        response=rag_qa(user_input)
        display(Markdown(f"Assistant: {response}"))

Human: whta is ml

Assistant: Machine learning is a branch of artificial intelligence that enables algorithms to uncover hidden patterns within datasets, allowing them to make predictions on new, similar data without explicit programming for each task. It combines data with statistical tools to predict outputs, yielding actionable insights, and finds applications in diverse fields such as image and speech recognition, natural language processing, recommendation systems, fraud detection, portfolio optimization, and automating tasks.

Human: explain ml and it types

Assistant: Machine learning is a branch of artificial intelligence that enables algorithms to uncover hidden patterns within datasets, allowing them to make predictions on new, similar data without explicit programming for each task. There are several types of machine learning, including:

1. **Supervised Learning**: This type of learning involves training a model on labeled data, where the correct output is already known. The goal is to learn a mapping between input data and the corresponding output labels, so the model can make predictions on new, unseen data.

2. **Unsupervised Learning**: This type of learning involves training a model on unlabeled data, where the model must find patterns or structure in the data on its own. Unsupervised learning can be further classified into two types:
   * **Clustering**: This technique is used to group similar data points into clusters, such that the data points within a cluster are more similar to each other than to data points in other clusters.
   * **Association**: This technique is used to find relationships between different variables in the data.

3. **Semi-Supervised Learning**: This type of learning lies between supervised and unsupervised learning, where the model is trained on a combination of labeled and unlabeled data. This approach is useful when labeled data is scarce or expensive to obtain.

To retrieve similar documents from a vector store, you can use machine learning algorithms such as:

* **K-Nearest Neighbors (KNN)**: This algorithm finds the k most similar documents to a given query document based on their vector representations.
* **Clustering**: This algorithm groups similar documents together into clusters, allowing you to retrieve all documents in a particular cluster.
* **Dimensionality Reduction**: This algorithm reduces the dimensionality of the vector representations, allowing for more efficient similarity searches.

Some popular machine learning frameworks for building these models include Scikit-learn, TensorFlow, and PyTorch. Additionally, big data tools like Hadoop and Spark can be used to process and store large amounts of data, while cloud platforms like AWS, Google Cloud, and Microsoft Azure provide scalable infrastructure for deploying machine learning models.

In the context of retrieving similar documents from a vector store, machine learning can be used to:

* **Learn vector representations**: Machine learning algorithms can be used to learn vector representations of documents, such that similar documents are closer together in the vector space.
* **Improve search efficiency**: Machine learning algorithms can be used to improve the efficiency of search queries, by reducing the dimensionality of the vector space or by using approximate nearest neighbor search algorithms.
* **Enhance search accuracy**: Machine learning algorithms can be used to enhance the accuracy of search results, by using techniques such as relevance feedback or query expansion.

Human: what is science

Assistant: To address your query about the definition of science and its various branches to retrieve relevant documents from the vector store, I must clarify that the provided context primarily focuses on Data Science, its definition, key components, and its applications, rather than the broader field of science and its branches.

However, I can provide a general overview of science and its main branches, which might help in understanding how Data Science fits into the broader scientific landscape.

Science is a systematic enterprise that builds and organizes knowledge in the form of testable explanations and predictions about the universe. The main branches of science are:

1. **Natural Sciences**: These include biology, chemistry, physics, and Earth sciences. They study the natural world around us, from the smallest subatomic particles to the entire cosmos.
2. **Formal Sciences**: This branch consists of mathematics, logic, and statistics. It deals with formal systems, including mathematical theories and logical deductions.
3. **Social Sciences**: These sciences study human behavior, societies, and the relationships among individuals within those societies. They include psychology, sociology, economics, and political science.

Data Science, as discussed in the provided context, is an interdisciplinary field that uses scientific methods, processes, algorithms, and systems to extract knowledge and insights from structured and unstructured data. It combines elements from natural sciences (like computer science), formal sciences (like statistics and mathematics), and social sciences (when analyzing human behavior through data).

To retrieve relevant documents from a vector store related to the definition of science and its various branches, you would need to query the database with keywords related to these branches or specific topics within them. However, without a direct connection to a database or more specific details about the query system you're using, I can only provide general guidance on how to approach such a query:

- **Query Example**: If you're looking for documents about the natural sciences, your query might include keywords like "biology," "chemistry," "physics," or "Earth sciences."
- **Filtering Results**: Depending on the capabilities of your vector store, you might be able to filter results by relevance, date, or specific categories within the natural, formal, or social sciences.

Since the provided context does not directly address the broader definition of science and its branches or provide instructions on how to query a vector store for such information, I hope this general overview is helpful. For more specific guidance on querying a vector store, additional details about the store's structure and query language would be necessary.

Human: what is thermodynamics

Assistant: I am sorry I don't have that information. The provided context does not relate to thermodynamics definition and principles to retrieve similar documents from a vector store. It appears to be related to machine learning concepts, such as linear regression, reinforcement learning, and metrics used in these fields. If you have any further questions or need information on a different topic, I'll be happy to help.

Human: exit